In [4]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Đo tài nguyên và tính cỡ mẫu huấn luyện tối đa (DPL302m, Task 1)

Mục đích: **tính** số mẫu huấn luyện `N_max` từ tài nguyên đo được trên Colab, thay vì đoán.

Nhãn giả `en_text` **đã tạo xong** (`LABELS_DONE = True`), nên chi phí mỗi mẫu gồm hai phần:

| Bước | Ký hiệu | Cách đo |
|---|---|---|
| Nhận dạng giọng nói (PhoWhisper) | `c_asr` | giây mỗi câu audio VietSpeech, đo trực tiếp (mục 3) |
| Fine-tune OPUS-MT | `E · F · t / bs` | `t` = giây mỗi bước, `bs` = batch size lớn nhất chạy được (mục 4) |

`c = c_asr + E · F · t / bs`  (nếu `LABELS_DONE = False` thì cộng thêm `1/r`, đo ở mục 2)

- `E` = số epoch; `F` = tổng các tỉ lệ cỡ mẫu của mọi lần fine-tune (ví dụ tập con 1/4, 1/2, 1 cho dữ liệu có lỗi ASR và 1 cho bản ghi gốc: `F = 2.75`).
- Nhận dạng chỉ chạy **một lần** cho `N_max` mẫu; các lần fine-tune dùng **tập con lồng nhau** của tập đó.
- **`N_max = (SAFETY · T − T_cố_định) / c`**, giới hạn trên bởi số mẫu có đủ `vi_text`, `en_text` **và file audio tồn tại** (nhận dạng cần audio).
- Ngân sách `T = N_SESSIONS × (SESSION_H − OVERHEAD_H)` khi chạy nhiều phiên từ checkpoint.

Mỗi mục ghi kết quả vào `measurements.json` trên Drive; mục 5 đọc lại file đó.

**Mô hình được lưu lại trên Drive** (`MODELS_DRIVE`) dưới dạng thư mục đầy đủ, không dùng cache Hugging Face trên Drive (cache dùng symlink, dễ hỏng trên Drive). Mỗi phiên chỉ cần sao chép từ Drive về ổ Colab (mục 0b).
Số đo chỉ đúng cho **loại GPU và phiên Colab bạn chạy** (ghi ở mục 0). **Cách dùng:** Runtime → T4 GPU; chạy ô cài đặt, mục 0–5 theo thứ tự.

### Tóm tắt công thức và kết quả tính cỡ mẫu huấn luyện tối đa ($N_{max}$)

#### 1. Công thức tính toán chính
- **Chi phí tổng cộng cho mỗi mẫu ($c$):**
  $$c = c_{label} + c_{asr} + c_{ft}$$
  *Trong đó:*
  - $c_{label}$: Chi phí tạo nhãn giả ($c_{label} = 0.0$ giây nếu `LABELS_DONE = True`, ngược lại bằng $1/r$ với $r$ là số câu dịch được mỗi giây ở Mục 2).
  - $c_{asr}$: Thời gian nhận dạng giọng nói trung bình mỗi câu bằng PhoWhisper (đo thực tế ở Mục 3).
  - $c_{ft}$: Chi phí fine-tune mô hình OPUS-MT quy về mỗi mẫu ($c_{ft} = \frac{E \cdot F \cdot t}{bs}$).
- **Cỡ mẫu lý thuyết tối đa ($N_{raw}$):**
  $$N_{raw} = \frac{\text{SAFETY} \cdot T - T_{c\hat{o}\,\ddot{d}"inh}}{c}$$
- **Cỡ mẫu sử dụng thực tế ($N_{max}$):**
  $$N_{max} = \min(N_{raw}, N_{available})$$
  *(Khi toàn bộ dữ liệu audio được tải lên đầy đủ, $N_{available} = 100,000$. Nếu tài nguyên giới hạn, $N_{raw}$ sẽ đóng vai trò quyết định)*

---

#### 2. Ý nghĩa các tham số trong tính toán tự động
- $T$: Ngân sách thời gian (tính bằng giờ).
- $E$: Số epoch fine-tune.
- $F$: Hệ số nhân tổng kích thước dữ liệu huấn luyện tích lũy qua các lần chạy con.
- $t$: Thời gian trung bình cho mỗi bước huấn luyện thực tế.
- $bs$: Batch size tối ưu được hệ thống lựa chọn dựa trên giới hạn an toàn của VRAM.

In [5]:
!pip -q install -U transformers accelerate sentencepiece sacremoses soundfile librosa "datasets<4"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 60.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 867.8/867.8 kB 52.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.6/294.6 kB 25.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 41.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.7/146.7 kB 16.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
gcsfs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2025.3.0 which is incompatible.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.3.1 which is incompatible.


## 0. Cấu hình

In [6]:
import os, io, gc, json, time, random, platform, traceback, itertools, shutil
import numpy as np, pandas as pd, torch, soundfile as sf, librosa
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from huggingface_hub import snapshot_download

# ---------- ĐƯỜNG DẪN ----------
DATA_DIR      = '/content/drive/MyDrive/project_DPL/datasets/vietspeech_100k'
METADATA_CSV  = f'{DATA_DIR}/metadata_with_en_vinai.csv'   # cột cần có: audio_path, vi_text, en_text
AUDIO_ROOT    = DATA_DIR                                    # audio_path tương đối so với thư mục này (có thư mục audio/)
OUT_DIR       = '/content/drive/MyDrive/project_DPL/checking_loading/measure_resource'
MODELS_DRIVE  = '/content/drive/MyDrive/project_DPL/models'   # bản lưu lâu dài của các mô hình
MODELS_LOCAL  = '/content/models'                              # bản làm việc trên ổ Colab (nhanh)
HF_CACHE      = '/content/hf_cache'                            # cache tạm trên ổ Colab (không đặt trên Drive)

# ---------- MÔ HÌNH ----------
ASR_MODEL     = 'vinai/PhoWhisper-small'
LABEL_MODEL   = 'vinai/vinai-translate-vi2en-v2'   # chỉ dùng nếu LABELS_DONE = False
MT_MODEL      = 'Helsinki-NLP/opus-mt-vi-en'

# ---------- THIẾT LẬP ĐO ----------
LABELS_DONE     = True    # True: en_text đã tạo xong cho các dòng hợp lệ -> bỏ chi phí tạo nhãn
N_MEASURE_ASR   = 200     # số câu audio đo tốc độ nhận dạng
N_MEASURE_LABEL = 500     # chỉ dùng khi LABELS_DONE = False
LABEL_BATCH     = 16
LABEL_BEAMS     = 4
ASR_BATCH       = 8
FT_BATCH_CANDIDATES = [8, 16, 32, 64, 128]
FT_BS_FORCE     = None    # None: chọn batch có thông lượng tốt nhất trong ngưỡng VRAM; hoặc nhập batch size nhóm định dùng khi huấn luyện
FT_MAXLEN       = 128
FT_STEPS        = 10      # số bước đo mỗi batch size (sau 2 bước khởi động)
SEED            = 42

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE  = torch.float16 if DEVICE == 'cuda' else torch.float32
SR     = 16000
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# try:
#     from google.colab import drive
#     drive.mount('/content/drive')
# except Exception as e:
#     print('Không gắn được Drive (bỏ qua nếu không chạy trên Colab):', e)
for d in (OUT_DIR, MODELS_DRIVE, MODELS_LOCAL, HF_CACHE): os.makedirs(d, exist_ok=True)
MEASURE_JSON = f'{OUT_DIR}/measurements.json'

def load_measures():
    return json.load(open(MEASURE_JSON)) if os.path.exists(MEASURE_JSON) else {}
def save_measures(upd):
    m = load_measures(); m.update(upd)
    json.dump(m, open(MEASURE_JSON, 'w'), indent=2, ensure_ascii=False)
    print('Đã ghi', MEASURE_JSON)

def free():
    gc.collect()
    if DEVICE == 'cuda': torch.cuda.empty_cache()
def sync():
    if DEVICE == 'cuda': torch.cuda.synchronize()

gpu = torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'CPU'
vram = torch.cuda.get_device_properties(0).total_memory / 2**30 if DEVICE == 'cuda' else 0
print('device:', DEVICE, '|', gpu, f'| VRAM {vram:.1f} GB')
if DEVICE != 'cuda': print('CẢNH BÁO: không có GPU, số đo không đại diện cho T4.')
save_measures({'env': {'gpu': gpu, 'vram_gb': round(vram, 1), 'torch': torch.__version__, 'python': platform.python_version()}})

device: cuda | Tesla T4 | VRAM 14.6 GB
Đã ghi /content/drive/MyDrive/project_DPL/checking_loading/measure_resource/measurements.json


## 0b. Tải và lưu các mô hình (một lần; phiên sau chỉ sao chép từ Drive)
Mỗi mô hình được tải bằng `snapshot_download(local_dir=...)` về ổ Colab, rồi sao chép sang `MODELS_DRIVE`. Từ phiên sau, ô này chỉ sao chép từ Drive về ổ Colab (không tải lại từ mạng). Thư mục chỉ được coi là hoàn tất khi có file `.complete`, nên lần tải dở dang sẽ tự tải lại. Nếu một mô hình bị hỏng, đặt `force=True` để xóa bản cũ và tải lại.

In [7]:
_BASE_PATTERNS = ['*.json', '*.txt', '*.model', '*.spm', '*.vocab', '*.bpe', '*.yml', '*.yaml']

def _complete(d):
    return os.path.isfile(os.path.join(d, 'config.json')) and os.path.isfile(os.path.join(d, '.complete'))

def _has_weights(d):
    return any(f.endswith(('.safetensors', '.bin')) for f in os.listdir(d))

def _dir_mb(d):
    return sum(os.path.getsize(os.path.join(r, f)) for r, _, fs in os.walk(d) for f in fs) / 2**20

def ensure_model(repo_id, force=False):
    name = repo_id.replace('/', '__')
    local, drv = os.path.join(MODELS_LOCAL, name), os.path.join(MODELS_DRIVE, name)
    if force:
        for d in (local, drv): shutil.rmtree(d, ignore_errors=True)
    if _complete(local): return local
    if _complete(drv):                                       # phiên mới: chỉ sao chép từ Drive
        shutil.rmtree(local, ignore_errors=True)
        shutil.copytree(drv, local, symlinks=False)
        print(f'[{repo_id}] sao chép từ Drive -> ổ Colab ({_dir_mb(local):.0f} MB)')
        return local
    shutil.rmtree(local, ignore_errors=True)                 # tải mới
    print(f'[{repo_id}] tải từ Hugging Face...')
    snapshot_download(repo_id, local_dir=local, allow_patterns=_BASE_PATTERNS + ['*.safetensors'])
    if not _has_weights(local):                              # mô hình chỉ có trọng số .bin
        snapshot_download(repo_id, local_dir=local, allow_patterns=_BASE_PATTERNS + ['*.bin'])
    assert _has_weights(local), f'{repo_id}: không thấy file trọng số'
    open(os.path.join(local, '.complete'), 'w').write('ok')
    tmp = drv + '.tmp'; shutil.rmtree(tmp, ignore_errors=True); shutil.rmtree(drv, ignore_errors=True)
    shutil.copytree(local, tmp, symlinks=False, ignore=shutil.ignore_patterns('.cache'))
    os.rename(tmp, drv)                                      # đổi tên sau cùng: không để lại bản dở trên Drive
    print(f'[{repo_id}] đã lưu lên Drive: {drv} ({_dir_mb(drv):.0f} MB)')
    return local

MODEL_PATHS = {}
for _repo in [ASR_MODEL, MT_MODEL] + ([] if LABELS_DONE else [LABEL_MODEL]):
    try:
        MODEL_PATHS[_repo] = ensure_model(_repo)
    except Exception as e:
        traceback.print_exc(); print(f'Không chuẩn bị được {_repo}:', e)
print('\nMô hình sẵn sàng:', {k: v for k, v in MODEL_PATHS.items()})

[vinai/PhoWhisper-small] sao chép từ Drive -> ổ Colab (926 MB)
[Helsinki-NLP/opus-mt-vi-en] sao chép từ Drive -> ổ Colab (278 MB)

Mô hình sẵn sàng: {'vinai/PhoWhisper-small': '/content/models/vinai__PhoWhisper-small', 'Helsinki-NLP/opus-mt-vi-en': '/content/models/Helsinki-NLP__opus-mt-vi-en'}


## 1. Nạp metadata, kiểm tra nhãn và đường dẫn audio
`audio_path` trong metadata dùng dấu `\` kiểu Windows (`audio\vs_000000.audio`); hàm `resolve_audio_path` đổi sang `/` và thử thêm `AUDIO_ROOT/audio/<tên file>`. Nếu không tìm thấy file nào, ô sẽ dừng và in lý do thay vì chuyển sang dữ liệu khác.

In [8]:
def resolve_audio_path(p):
    p = str(p).replace('\\', '/')
    cands = [p] if os.path.isabs(p) else [os.path.join(AUDIO_ROOT, p)]
    cands.append(os.path.join(AUDIO_ROOT, 'audio', os.path.basename(p)))
    for c in cands:
        if os.path.exists(c): return c
    return None

df = pd.read_csv(METADATA_CSV)
n_rows = len(df)
print('Số dòng:', n_rows, '| cột:', list(df.columns))
for col in ['audio_path', 'vi_text']:
    assert col in df.columns, f'Thiếu cột {col}'
ok = df.vi_text.notna() & (df.vi_text.astype(str).str.strip() != '')
if 'en_text' in df.columns:
    has_en_row = df.en_text.notna() & (df.en_text.astype(str).str.strip() != '')
    print(f'Dòng có en_text không rỗng: {int(has_en_row.sum()):,} / {n_rows:,}')
    if LABELS_DONE: ok = ok & has_en_row
elif LABELS_DONE:
    raise AssertionError('LABELS_DONE=True nhưng metadata không có cột en_text')
df = df[ok].reset_index(drop=True)
N_LABELED = len(df)
HAS_EN = 'en_text' in df.columns
print('Số mẫu có vi_text + nhãn en_text (N_LABELED):', f'{N_LABELED:,}')
assert N_LABELED > 0, 'Không có mẫu hợp lệ.'

# Lấy mẫu ngẫu nhiên dựa trên SEED để kiểm tra
N_PROBE = 1000
rng = np.random.RandomState(SEED)
probe_idx = rng.choice(N_LABELED, size=min(N_PROBE, N_LABELED), replace=False)

t0 = time.time(); hit = np.zeros(len(probe_idx), dtype=bool)
for k, i in enumerate(probe_idx):
    try: hit[k] = resolve_audio_path(df.audio_path.iloc[i]) is not None
    except OSError: hit[k] = False
    if k == 49: print(f'50 file đầu mất {time.time()-t0:.0f}s -> ước tính cả {len(probe_idx)} file ~{(time.time()-t0)*len(probe_idx)/50/60:.1f} phút')
n_hit, n_pr = int(hit.sum()), len(probe_idx)
p = n_hit / n_pr
z = 1.96; den = 1 + z*z/n_pr
ctr = (p + z*z/(2*n_pr)) / den; hw = z * np.sqrt(p*(1-p)/n_pr + z*z/(4*n_pr*n_pr)) / den
p_lo, p_hi = max(0.0, ctr - hw), min(1.0, ctr + hw)
print(f'Kiểm tra {n_pr} dòng ngẫu nhiên ({time.time()-t0:.0f}s): có audio {n_hit} ({p:.1%}); khoảng tin cậy 95%: {p_lo:.1%} - {p_hi:.1%}')

# Tỉ lệ có audio theo từng khúc chỉ số dòng
bins = np.array_split(np.argsort(probe_idx), 10)
print('Tỉ lệ có audio theo 10 khúc chỉ số (đầu -> cuối):', [f'{hit[b].mean():.0%}' for b in bins])
assert n_hit >= 10, f'Chỉ {n_hit} file có audio; kiểm tra AUDIO_ROOT={AUDIO_ROOT}'

N_AVAILABLE    = int(p * N_LABELED)
N_AVAILABLE_LO = int(p_lo * N_LABELED)
print(f'Ước lượng N_AVAILABLE ≈ {N_AVAILABLE:,} (cận dưới 95%: {N_AVAILABLE_LO:,}) / {n_rows:,}')

# Mẫu đo nhận dạng: lấy các dòng đã xác nhận có tồn tại file audio từ tập mẫu thử
have_audio = probe_idx[hit]
idx_asr   = rng.choice(have_audio, size=min(N_MEASURE_ASR, len(have_audio)), replace=False)
# Tạo nhãn không cần audio: lấy ngẫu nhiên
idx_label = rng.choice(N_LABELED, size=min(N_MEASURE_LABEL, N_LABELED), replace=False)

save_measures({'data': {'n_rows': int(n_rows), 'n_labeled': int(N_LABELED), 'n_available': int(N_AVAILABLE),
                        'n_available_lo': int(N_AVAILABLE_LO), 'available_exact': False, 'probe_n': int(n_pr), 'probe_hit': n_hit,
                        'labels_done': LABELS_DONE, 'metadata_csv': METADATA_CSV, 'seed': SEED}})

Số dòng: 100000 | cột: ['id', 'source_index', 'audio_path', 'vi_text', 'sampling_rate', 'audio_format', 'en_text']
Dòng có en_text không rỗng: 100,000 / 100,000
Số mẫu có vi_text + nhãn en_text (N_LABELED): 100,000
50 file đầu mất 278s -> ước tính cả 1000 file ~92.6 phút
Kiểm tra 1000 dòng ngẫu nhiên (278s): có audio 999 (99.9%); khoảng tin cậy 95%: 99.4% - 100.0%
Tỉ lệ có audio theo 10 khúc chỉ số (đầu -> cuối): ['100%', '100%', '100%', '100%', '100%', '100%', '100%', '99%', '100%', '100%']
Ước lượng N_AVAILABLE ≈ 99,900 (cận dưới 95%: 99,435) / 100,000
Đã ghi /content/drive/MyDrive/project_DPL/checking_loading/measure_resource/measurements.json


In [9]:
def resolve_audio_path(p):
    p = str(p).replace('\\', '/')
    cands = [p] if os.path.isabs(p) else [os.path.join(AUDIO_ROOT, p)]
    cands.append(os.path.join(AUDIO_ROOT, 'audio', os.path.basename(p)))
    for c in cands:
        if os.path.exists(c): return c
    return None

df = pd.read_csv(METADATA_CSV)
n_rows = len(df)
print('Số dòng:', n_rows, '| cột:', list(df.columns))
for col in ['audio_path', 'vi_text']:
    assert col in df.columns, f'Thiếu cột {col}'
ok = df.vi_text.notna() & (df.vi_text.astype(str).str.strip() != '')
if 'en_text' in df.columns:
    has_en_row = df.en_text.notna() & (df.en_text.astype(str).str.strip() != '')
    print(f'Dòng có en_text không rỗng: {int(has_en_row.sum()):,} / {n_rows:,}')
    if LABELS_DONE: ok = ok & has_en_row
elif LABELS_DONE:
    raise AssertionError('LABELS_DONE=True nhưng metadata không có cột en_text')
df = df[ok].reset_index(drop=True)
N_LABELED = len(df)
HAS_EN = 'en_text' in df.columns
print('Số mẫu có vi_text + nhãn en_text (N_LABELED):', f'{N_LABELED:,}')
assert N_LABELED > 0, 'Không có mẫu hợp lệ.'

# Không dùng os.listdir() (lỗi Input/output error trên Drive khi thư mục quá lớn).
# Ước lượng tỉ lệ dòng có file audio bằng cách kiểm tra os.path.exists trên mẫu ngẫu nhiên.
N_PROBE = 1000
rng = np.random.RandomState(SEED)
probe_idx = rng.choice(N_LABELED, size=min(N_PROBE, N_LABELED), replace=False)
t0 = time.time(); hit = np.zeros(len(probe_idx), dtype=bool)
for k, i in enumerate(probe_idx):
    try: hit[k] = resolve_audio_path(df.audio_path.iloc[i]) is not None
    except OSError: hit[k] = False
    if k == 49: print(f'50 file đầu mất {time.time()-t0:.0f}s -> ước tính cả {len(probe_idx)} file ~{(time.time()-t0)*len(probe_idx)/50/60:.1f} phút')
n_hit, n_pr = int(hit.sum()), len(probe_idx)
p = n_hit / n_pr
z = 1.96; den = 1 + z*z/n_pr                       # khoảng tin cậy Wilson 95%
ctr = (p + z*z/(2*n_pr)) / den; hw = z * np.sqrt(p*(1-p)/n_pr + z*z/(4*n_pr*n_pr)) / den
p_lo, p_hi = max(0.0, ctr - hw), min(1.0, ctr + hw)
print(f'Kiểm tra {n_pr} dòng ngẫu nhiên ({time.time()-t0:.0f}s): có audio {n_hit} ({p:.1%}); khoảng tin cậy 95%: {p_lo:.1%} - {p_hi:.1%}')
# Tỉ lệ có audio theo từng khúc chỉ số dòng, để biết file thiếu là ngẫu nhiên hay theo vùng
bins = np.array_split(np.argsort(probe_idx), 10)
print('Tỉ lệ có audio theo 10 khúc chỉ số (đầu -> cuối):', [f'{hit[b].mean():.0%}' for b in bins])
assert n_hit >= 10, f'Chỉ {n_hit} file có audio; kiểm tra AUDIO_ROOT={AUDIO_ROOT}'

N_AVAILABLE    = int(p * N_LABELED)               # ƯỚC LƯỢNG số mẫu có nhãn VÀ audio
N_AVAILABLE_LO = int(p_lo * N_LABELED)            # cận dưới 95%, dùng làm giới hạn trên an toàn của N_max
print(f'Ước lượng N_AVAILABLE ≈ {N_AVAILABLE:,} (cận dưới 95%: {N_AVAILABLE_LO:,}) / {n_rows:,}  [KHÔNG phải số đếm chính xác]')

# Mẫu đo nhận dạng: chỉ lấy dòng đã xác nhận có audio (để không phí lượt đo vào file thiếu)
have_audio = probe_idx[hit]
idx_asr   = rng.choice(have_audio, size=min(N_MEASURE_ASR, len(have_audio)), replace=False)
idx_label = rng.choice(N_LABELED, size=min(N_MEASURE_LABEL, N_LABELED), replace=False)   # tạo nhãn không cần audio
save_measures({'data': {'n_rows': int(n_rows), 'n_labeled': int(N_LABELED), 'n_available': int(N_AVAILABLE),
                        'n_available_lo': int(N_AVAILABLE_LO), 'available_exact': False, 'probe_n': int(n_pr), 'probe_hit': n_hit,
                        'labels_done': LABELS_DONE, 'metadata_csv': METADATA_CSV, 'seed': SEED}})

Số dòng: 100000 | cột: ['id', 'source_index', 'audio_path', 'vi_text', 'sampling_rate', 'audio_format', 'en_text']
Dòng có en_text không rỗng: 100,000 / 100,000
Số mẫu có vi_text + nhãn en_text (N_LABELED): 100,000
50 file đầu mất 0s -> ước tính cả 1000 file ~0.0 phút
Kiểm tra 1000 dòng ngẫu nhiên (0s): có audio 1000 (100.0%); khoảng tin cậy 95%: 99.6% - 100.0%
Tỉ lệ có audio theo 10 khúc chỉ số (đầu -> cuối): ['100%', '100%', '100%', '100%', '100%', '100%', '100%', '100%', '100%', '100%']
Ước lượng N_AVAILABLE ≈ 100,000 (cận dưới 95%: 99,617) / 100,000  [KHÔNG phải số đếm chính xác]
Đã ghi /content/drive/MyDrive/project_DPL/checking_loading/measure_resource/measurements.json


## 2. (Chỉ khi `LABELS_DONE = False`) Đo tốc độ tạo nhãn giả VinAI (`r`, câu/giây)

In [10]:
def measure_labeling():
    free()
    if DEVICE == 'cuda': torch.cuda.reset_peak_memory_stats()
    lp = MODEL_PATHS.get(LABEL_MODEL) or ensure_model(LABEL_MODEL)
    tok = AutoTokenizer.from_pretrained(lp, src_lang='vi_VN')
    m = AutoModelForSeq2SeqLM.from_pretrained(lp).to(DEVICE)
    if DEVICE == 'cuda': m = m.half()
    m.eval()
    tgt = tok.convert_tokens_to_ids('en_XX')
    texts = df.vi_text.astype(str).iloc[idx_label].tolist()
    def run(batch):
        b = tok(batch, return_tensors='pt', padding=True, truncation=True, max_length=256).to(DEVICE)
        with torch.no_grad():
            out = m.generate(**b, decoder_start_token_id=tgt, num_beams=LABEL_BEAMS, early_stopping=True, max_new_tokens=256)
        return tok.batch_decode(out, skip_special_tokens=True)
    run(texts[:LABEL_BATCH]); sync()
    t0 = time.perf_counter(); outs = []
    for i in range(0, len(texts), LABEL_BATCH): outs += run(texts[i:i+LABEL_BATCH])
    sync(); el = time.perf_counter() - t0
    peak = torch.cuda.max_memory_allocated() / 2**30 if DEVICE == 'cuda' else None
    del m; free()
    return {'n': len(texts), 'seconds': round(el, 2), 'r_sent_per_s': round(len(texts) / el, 2),
            'batch': LABEL_BATCH, 'beams': LABEL_BEAMS, 'peak_vram_gb': None if peak is None else round(peak, 2),
            'mean_chars': round(float(np.mean([len(x) for x in texts])), 1), 'example': outs[0][:120]}

if LABELS_DONE:
    print('LABELS_DONE=True: bỏ qua mục 2 (chi phí tạo nhãn = 0).')
else:
    try:
        res = measure_labeling(); print(res); save_measures({'labeling': res})
    except Exception as e:
        traceback.print_exc(); print('Mục 2 lỗi:', e)

LABELS_DONE=True: bỏ qua mục 2 (chi phí tạo nhãn = 0).


## 3. Đo tốc độ nhận dạng PhoWhisper (`c_asr`, giây mỗi câu; `RTF`)
Dùng audio VietSpeech thật. Ô này **không** chuyển sang dữ liệu khác khi đọc audio lỗi: nó in số file lỗi theo từng lý do rồi dừng.

In [11]:
def read_wav_16k(path):
    try:
        w, sr = sf.read(path, dtype='float32')
    except Exception:
        w, sr = librosa.load(path, sr=None, mono=False)       # phương án dự phòng cho định dạng lạ
        w = np.asarray(w, dtype='float32')
        if w.ndim > 1: w = w.mean(axis=0)
    if w.ndim > 1: w = w.mean(axis=1)
    if sr != SR: w = librosa.resample(w, orig_sr=sr, target_sr=SR)
    return w

def load_audio_files(idxs):
    wavs, durs, fails = [], [], {'not_found': 0, 'read_error': 0}
    err_examples = []
    for i in idxs:
        p = resolve_audio_path(df.audio_path.iloc[i])
        if p is None:
            fails['not_found'] += 1; continue
        try:
            w = read_wav_16k(p)
        except Exception as e:
            fails['read_error'] += 1
            if len(err_examples) < 3: err_examples.append(f'{p}: {type(e).__name__}: {str(e)[:80]}')
            continue
        if len(w) == 0:
            fails['read_error'] += 1; continue
        durs.append(len(w) / SR); wavs.append(w[:SR * 30])
    return wavs, durs, fails, err_examples

def build_asr_pipeline():
    from transformers import pipeline
    def make(path):
        kw = dict(model=path, device=0 if DEVICE == 'cuda' else -1)
        try: return pipeline('automatic-speech-recognition', dtype=DTYPE, **kw)
        except TypeError: return pipeline('automatic-speech-recognition', torch_dtype=DTYPE, **kw)
    try:
        return make(MODEL_PATHS.get(ASR_MODEL) or ensure_model(ASR_MODEL))
    except Exception as e:
        print('Lỗi nạp PhoWhisper từ bản đã lưu, xóa và tải lại bản sạch:', type(e).__name__, str(e)[:100])
        return make(ensure_model(ASR_MODEL, force=True))

def asr_items(ws):
    # pipeline lấy (pop) 'raw' và 'sampling_rate' khỏi dict nên phải tạo dict MỚI cho mỗi lần gọi
    return [{'raw': w, 'sampling_rate': SR} for w in ws]

def measure_asr():
    wavs, durs, fails, err_examples = load_audio_files(idx_asr)
    print(f'Đọc được {len(wavs)}/{len(idx_asr)} file audio | lỗi: {fails}')
    if len(wavs) < 10:
        raise RuntimeError(f'Chỉ đọc được {len(wavs)} file audio (lỗi: {fails}). Ví dụ lỗi đọc: {err_examples}')
    free()
    if DEVICE == 'cuda': torch.cuda.reset_peak_memory_stats()
    pipe = build_asr_pipeline()
    gk = {'language': 'vi', 'task': 'transcribe'}
    pipe(asr_items(wavs[:2]), batch_size=2, generate_kwargs=gk); sync()   # khởi động, không tính giờ
    t0 = time.perf_counter(); out = pipe(asr_items(wavs), batch_size=ASR_BATCH, generate_kwargs=gk); sync()
    el = time.perf_counter() - t0
    peak = torch.cuda.max_memory_allocated() / 2**30 if DEVICE == 'cuda' else None
    tot_audio = float(sum(min(d, 30) for d in durs))
    del pipe; free()
    return {'source': 'VietSpeech (metadata.audio_path)', 'n': len(wavs), 'files_failed': fails,
            'seconds': round(el, 2), 'c_asr_s_per_utt': round(el / len(wavs), 3),
            'mean_dur_s': round(float(np.mean(durs)), 2), 'rtf': round(el / tot_audio, 4), 'batch': ASR_BATCH,
            'peak_vram_gb': None if peak is None else round(peak, 2), 'example': out[0]['text'][:120]}

try:
    res = measure_asr(); print(res); save_measures({'asr': res})
except Exception as e:
    traceback.print_exc(); print('Mục 3 lỗi:', e)

Đọc được 200/200 file audio | lỗi: {'not_found': 0, 'read_error': 0}


Loading weights:   0%|          | 0/480 [00:00<?, ?it/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'suppress_tokens', 'begin_suppress_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer WhisperTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


{'source': 'VietSpeech (metadata.audio_path)', 'n': 200, 'files_failed': {'not_found': 0, 'read_error': 0}, 'seconds': 63.46, 'c_asr_s_per_utt': 0.317, 'mean_dur_s': 3.85, 'rtf': 0.0825, 'batch': 8, 'peak_vram_gb': 2.84, 'example': 'và một túi ra để đựng mọi thứ.'}
Đã ghi /content/drive/MyDrive/project_DPL/checking_loading/measure_resource/measurements.json


## 4. Đo fine-tune OPUS-MT: giây mỗi bước `t` và batch size
Nguồn `vi_text`, đích `en_text` thật (độ dài câu thật). fp32 trọng số + autocast fp16 trên GPU, AdamW, đệm tới câu dài nhất trong batch, độ dài tối đa `FT_MAXLEN`. Batch size gây hết bộ nhớ bị loại hoặc dùng quá 90% VRAM bị bỏ. Mặc định chọn batch có thông lượng tốt nhất (giây/mẫu thấp nhất); nếu nhóm đã định batch size huấn luyện thì đặt `FT_BS_FORCE`. Lưu ý batch lớn làm giảm số bước cập nhật trên mỗi epoch, nên chọn batch theo cả chất lượng huấn luyện chứ không chỉ tốc độ.

In [12]:
def make_scaler():
    try: return torch.amp.GradScaler('cuda', enabled=(DEVICE == 'cuda'))
    except Exception: return torch.cuda.amp.GradScaler(enabled=(DEVICE == 'cuda'))

def measure_finetune():
    free()
    mt_path = MODEL_PATHS.get(MT_MODEL) or ensure_model(MT_MODEL)
    tok = AutoTokenizer.from_pretrained(mt_path)
    pool = df.sample(n=min(4096, N_AVAILABLE), random_state=SEED)
    src = pool.vi_text.astype(str).tolist()
    if HAS_EN:
        tgt = pool.en_text.fillna('').astype(str).tolist()
        keep = [i for i, t in enumerate(tgt) if t.strip()]
        src = [src[i] for i in keep]; tgt = [tgt[i] for i in keep]; tgt_note = 'en_text thật'
    else:
        tgt = src; tgt_note = 'dùng nguồn làm đích (chỉ đo thời gian)'
    assert len(src) >= max(FT_BATCH_CANDIDATES), 'Không đủ mẫu để thử batch lớn nhất.'
    rows = []
    for bs in FT_BATCH_CANDIDATES:
        row = {'bs': bs, 'status': 'FAIL', 't_s_per_step': None, 'peak_vram_gb': None, 'note': ''}
        m = opt = None
        try:
            free()
            if DEVICE == 'cuda': torch.cuda.reset_peak_memory_stats()
            m = AutoModelForSeq2SeqLM.from_pretrained(mt_path).to(DEVICE).train()
            opt = torch.optim.AdamW(m.parameters(), lr=5e-5)
            scaler = make_scaler()
            rs = np.random.RandomState(SEED); times = []
            warm = 2 + (5 if bs == FT_BATCH_CANDIDATES[0] else 0)   # lần đầu có khởi động CUDA, cần thêm bước loại bỏ
            for step in range(warm + FT_STEPS):
                ids = rs.choice(len(src), size=bs, replace=False)
                b = tok([src[i] for i in ids], text_target=[tgt[i] for i in ids], return_tensors='pt',
                        padding=True, truncation=True, max_length=FT_MAXLEN).to(DEVICE)
                b['labels'][b['labels'] == tok.pad_token_id] = -100
                sync(); t0 = time.perf_counter()
                opt.zero_grad(set_to_none=True)
                with torch.autocast(device_type=DEVICE, dtype=torch.float16, enabled=(DEVICE == 'cuda')):
                    loss = m(**b).loss
                scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
                sync()
                if step >= warm: times.append(time.perf_counter() - t0)
            row['t_s_per_step'] = round(float(np.mean(times)), 4)
            row['peak_vram_gb'] = round(torch.cuda.max_memory_allocated() / 2**30, 2) if DEVICE == 'cuda' else None
            row['status'] = 'OK'
        except Exception as e:
            row['note'] = f'{type(e).__name__}: {str(e)[:100]}'
        finally:
            del m, opt; free()
        rows.append(row); print(row)
    ok_rows = [r for r in rows if r['status'] == 'OK' and (vram == 0 or (r['peak_vram_gb'] or 0) <= 0.9 * vram)]
    if FT_BS_FORCE:
        best = next((r for r in ok_rows if r['bs'] == FT_BS_FORCE), None)
        if best is None: print(f'CẢNH BÁO: FT_BS_FORCE={FT_BS_FORCE} không chạy được hoặc vượt 90% VRAM.')
    else:   # batch có thông lượng tốt nhất (giây/mẫu thấp nhất) trong ngưỡng VRAM
        best = min(ok_rows, key=lambda r: r['t_s_per_step'] / r['bs']) if ok_rows else None
    return {'rows': rows, 'target_note': tgt_note, 'maxlen': FT_MAXLEN,
            'chosen_bs': best['bs'] if best else None, 't_s_per_step': best['t_s_per_step'] if best else None,
            'sec_per_sample_per_epoch': round(best['t_s_per_step'] / best['bs'], 5) if best else None}

try:
    res = measure_finetune(); print({k: v for k, v in res.items() if k != 'rows'}); save_measures({'finetune': res})
except Exception as e:
    traceback.print_exc(); print('Mục 4 lỗi:', e)

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

{'bs': 8, 'status': 'OK', 't_s_per_step': 0.0754, 'peak_vram_gb': 1.36, 'note': ''}


Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

{'bs': 16, 'status': 'OK', 't_s_per_step': 0.0807, 'peak_vram_gb': 1.55, 'note': ''}


Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

{'bs': 32, 'status': 'OK', 't_s_per_step': 0.0959, 'peak_vram_gb': 2.25, 'note': ''}


Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

{'bs': 64, 'status': 'OK', 't_s_per_step': 0.1419, 'peak_vram_gb': 3.76, 'note': ''}


Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

{'bs': 128, 'status': 'OK', 't_s_per_step': 0.2537, 'peak_vram_gb': 7.35, 'note': ''}
{'target_note': 'en_text thật', 'maxlen': 128, 'chosen_bs': 128, 't_s_per_step': 0.2537, 'sec_per_sample_per_epoch': 0.00198}
Đã ghi /content/drive/MyDrive/project_DPL/checking_loading/measure_resource/measurements.json


## 5. Tính `N_max`
Các tham số dưới đây **do nhóm quyết định** (giá trị trong ô chỉ là giá trị khởi tạo cần thay bằng số thật của nhóm):

- `N_SESSIONS`, `SESSION_H`, `OVERHEAD_H`: số phiên Colab nhóm thực sự chạy được, số giờ mỗi phiên, và giờ mỗi phiên mất cho khởi động (gắn Drive, cài thư viện, nạp mô hình, khôi phục checkpoint). Chỉ cộng dồn các phiên nếu mọi bước chạy tiếp được từ checkpoint.
- `T_FIXED_H`: thời gian không phụ thuộc cỡ mẫu (nạp mô hình, đánh giá A/B/C trên tập test, System C, kiểm tra thủ công); ô này không tự đo.
- `FRACTIONS_NOISY`, `FRACTIONS_CLEAN`: tỉ lệ cỡ mẫu của từng lần fine-tune (tập con lồng nhau).

In [16]:
# ---------- THAM SỐ NHÓM QUYẾT ĐỊNH (thay bằng số thật của nhóm) ----------
N_SESSIONS      = 5                  # số phiên Colab thực sự chạy được trước hạn
SESSION_H       = 3.0                # số giờ mỗi phiên
OVERHEAD_H      = 0.3                # giờ khởi động mỗi phiên (nên đo thực tế)
T_BUDGET_H      = None               # None: tự tính = N_SESSIONS * (SESSION_H - OVERHEAD_H); hoặc nhập số giờ trực tiếp
T_FIXED_H       = 2.0                # giờ cố định (đánh giá, System C...): nhóm tự ước lượng
SAFETY          = 0.7                # hệ số an toàn
EPOCHS          = 30                 # số epoch fine-tune
FRACTIONS_NOISY = [0.25, 0.5, 1.0]   # tập con lồng nhau, dữ liệu có lỗi ASR
FRACTIONS_CLEAN = [1.0]              # đối chứng: bản ghi gốc
N_CAP           = None               # None: lấy theo N_AVAILABLE trong measurements.json

M = load_measures()
need = ['asr', 'finetune'] + ([] if LABELS_DONE else ['labeling'])
missing = [k for k in need if k not in M or not M[k]]
assert not missing, f'Thiếu số đo: {missing}. Chạy lại mục tương ứng.'
r      = None if LABELS_DONE else M['labeling']['r_sent_per_s']
c_asr  = M['asr']['c_asr_s_per_utt']
t_step = M['finetune']['t_s_per_step']
bs     = M['finetune']['chosen_bs']
assert bs, 'Không có batch size nào chạy được ở mục 4.'
n_avail = M['data'].get('n_available_lo', M['data']['n_available'])   # cận dưới 95% khi chỉ ước lượng
cap = N_CAP or n_avail
T_total_h = T_BUDGET_H if T_BUDGET_H is not None else N_SESSIONS * (SESSION_H - OVERHEAD_H)

F = sum(FRACTIONS_NOISY) + sum(FRACTIONS_CLEAN)
c_label = 0.0 if LABELS_DONE else 1 / r
c_ft    = EPOCHS * F * t_step / bs
c       = c_label + c_asr + c_ft
usable_s = SAFETY * T_total_h * 3600 - T_FIXED_H * 3600
N_raw   = max(0, usable_s / c)
N_max   = int(min(N_raw, cap))
N_nice  = (N_max // 1000) * 1000 if N_max >= 2000 else N_max

print('Nguồn số đo: GPU', M['env']['gpu'], '| ASR dùng', M['asr']['source'])
print('tạo nhãn giả: đã xong (chi phí 0)' if LABELS_DONE else f'r = {r} câu/s -> {c_label:.4f} s/mẫu (tạo nhãn)')
print(f'c_asr = {c_asr} s/mẫu (nhận dạng); RTF = {M["asr"]["rtf"]}; thời lượng TB = {M["asr"]["mean_dur_s"]} s')
print(f'fine-tune: bs={bs}, t={t_step} s/bước, F={F} lần-cỡ, E={EPOCHS} -> {c_ft:.4f} s/mẫu')
print(f'c tổng = {c:.4f} s/mẫu')
print(f'Ngân sách T = {T_total_h:.2f} h | thời gian khả dụng = {usable_s/3600:.2f} h')
print(f'\nN_max (chưa giới hạn) = {N_raw:,.0f} | số mẫu có nhãn + audio (cận dưới) = {n_avail:,} (có nhãn: {M["data"].get("n_labeled", "?")})')
print(f'=> N_max dùng được = {N_max:,}  (làm tròn: {N_nice:,})')

N = N_nice
print('\nTập con fine-tune (lồng nhau):')
print(' - có lỗi ASR :', [int(N * f) for f in FRACTIONS_NOISY])
print(' - bản ghi gốc:', [int(N * f) for f in FRACTIONS_CLEAN])
bd = pd.DataFrame({'bước': ['tạo nhãn giả (đã xong)' if LABELS_DONE else 'tạo nhãn giả', 'nhận dạng', 'fine-tune (mọi lần chạy)'],
                   'giờ': [N * c_label / 3600, N * c_asr / 3600, N * c_ft / 3600]})
bd['giờ'] = bd['giờ'].round(2)
display(bd) if 'display' in globals() else print(bd)
if N_raw < cap: print('Tài nguyên là giới hạn chính (N_max < số mẫu có nhãn và audio).')
else:           print('Dữ liệu là giới hạn chính (đã chạm số mẫu có nhãn và audio); có thể giảm ngân sách hoặc tăng số epoch.')

Nguồn số đo: GPU Tesla T4 | ASR dùng VietSpeech (metadata.audio_path)
tạo nhãn giả: đã xong (chi phí 0)
c_asr = 0.317 s/mẫu (nhận dạng); RTF = 0.0825; thời lượng TB = 3.85 s
fine-tune: bs=128, t=0.2537 s/bước, F=2.75 lần-cỡ, E=30 -> 0.1635 s/mẫu
c tổng = 0.4805 s/mẫu
Ngân sách T = 13.50 h | thời gian khả dụng = 7.45 h

N_max (chưa giới hạn) = 55,815 | số mẫu có nhãn + audio (cận dưới) = 99,617 (có nhãn: 100000)
=> N_max dùng được = 55,814  (làm tròn: 55,000)

Tập con fine-tune (lồng nhau):
 - có lỗi ASR : [13750, 27500, 55000]
 - bản ghi gốc: [55000]
                       bước   giờ
0    tạo nhãn giả (đã xong)  0.00
1                 nhận dạng  4.84
2  fine-tune (mọi lần chạy)  2.50
Tài nguyên là giới hạn chính (N_max < số mẫu có nhãn và audio).


### Bảng nhạy: `N_max` theo ngân sách giờ và số epoch

In [17]:
rows = []
for T in [4, 8, 12, 24, 48]:
    row = {'T (giờ)': T}
    for E in [1, 2, 3]:
        cE = c_label + c_asr + E * F * t_step / bs
        n = max(0, (SAFETY * T * 3600 - T_FIXED_H * 3600) / cE)
        row[f'E={E}'] = int(min(n, cap))
    rows.append(row)
tbl = pd.DataFrame(rows).set_index('T (giờ)')
display(tbl) if 'display' in globals() else print(tbl)
print('Bảng đã giới hạn bởi số mẫu có nhãn và audio, dùng cùng SAFETY, T_FIXED_H, FRACTIONS ở ô trên.')

           E=1    E=2    E=3
T (giờ)                     
4         8931   8783   8639
8        40192  39524  38877
12       71452  70265  69116
24       99617  99617  99617
48       99617  99617  99617
Bảng đã giới hạn bởi số mẫu có nhãn và audio, dùng cùng SAFETY, T_FIXED_H, FRACTIONS ở ô trên.


In [18]:
import json, os
import pandas as pd

# Tải dữ liệu đo lường thực tế
MEASURE_JSON = '/content/drive/MyDrive/project_DPL/checking_loading/measure_resource/measurements.json'
if os.path.exists(MEASURE_JSON):
    M = json.load(open(MEASURE_JSON))

    print("=== CÁC GIÁ TRỊ ĐO ĐƯỢC THỰC TẾ ===")
    print(f"- GPU Sử dụng: {M.get('env', {}).get('gpu', 'N/A')} ({M.get('env', {}).get('vram_gb', 0)} GB VRAM)")

    c_asr = M.get('asr', {}).get('c_asr_s_per_utt', 0)
    rtf = M.get('asr', {}).get('rtf', 0)
    mean_dur = M.get('asr', {}).get('mean_dur_s', 0)
    print(f"- Chi phí ASR (c_asr): {c_asr:.4f} giây/mẫu (RTF: {rtf:.4f}, Độ dài TB: {mean_dur:.2f}s)")

    labels_done = M.get('data', {}).get('labels_done', True)
    if labels_done:
        c_label = 0.0
        print("- Chi phí Labeling (c_label): 0.0000 giây/mẫu (Đã hoàn thành tạo nhãn trước)")
    else:
        r = M.get('labeling', {}).get('r_sent_per_s', 1.0)
        c_label = 1 / r
        print(f"- Chi phí Labeling (c_label): {c_label:.4f} giây/mẫu (Tốc độ dịch: {r:.2f} câu/s)")

    t_step = M.get('finetune', {}).get('t_s_per_step', 0)
    bs = M.get('finetune', {}).get('chosen_bs', 1)
    print(f"- Chi phí Fine-tune: t_step = {t_step:.4f} giây/bước | Batch Size tối ưu = {bs}")

    # In dữ liệu ước tính sẵn có
    n_avail = M.get('data', {}).get('n_available_lo', M.get('data', {}).get('n_available', 0))
    print(f"- Số mẫu khả dụng thực tế ước lượng (N_available cận dưới 95%): {n_avail:,} mẫu")
else:
    print("Không tìm thấy file measurements.json. Vui lòng chạy các mục đo lường trước.")

=== CÁC GIÁ TRỊ ĐO ĐƯỢC THỰC TẾ ===
- GPU Sử dụng: Tesla T4 (14.6 GB VRAM)
- Chi phí ASR (c_asr): 0.3170 giây/mẫu (RTF: 0.0825, Độ dài TB: 3.85s)
- Chi phí Labeling (c_label): 0.0000 giây/mẫu (Đã hoàn thành tạo nhãn trước)
- Chi phí Fine-tune: t_step = 0.2537 giây/bước | Batch Size tối ưu = 128
- Số mẫu khả dụng thực tế ước lượng (N_available cận dưới 95%): 99,617 mẫu
